# Hyperparameter Tunning  
Notebook for hyperparamater tuning code.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# keeps warnings from printing
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore')# suppress all warnings
# if you want more targeted:
# warnings.filterwarnings('ignore', category=FutureWarning)
# warnings.filterwarnings('ignore', category=ConvergenceWarning)

In [3]:
import pandas as pd
import numpy as np
from lib.superdataset_utils import load_dfs_subset, DFS_QUESTION_COLS
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import textwrap
import random
import sklearn
from sklearn.model_selection import RandomizedSearchCV, train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import \
    classification_report, cohen_kappa_score, balanced_accuracy_score, accuracy_score, f1_score, make_scorer
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import randint, uniform, loguniform, spearmanr
import xgboost as xgb
import lightgbm as lgb
import shap
from mord import LogisticAT
from catboost import CatBoostClassifier, CatBoostRegressor

In [4]:
print(f'sklearn version: {sklearn.__version__}')
print(f'xgb version: {xgb.__version__}')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')
print(f'shap version: {shap.__version__}')

sklearn version: 1.9.1
xgb version: 3.1.3
pandas version: 2.3.3
numpy version: 2.4.6
shap version: 0.50.0


---

## Load superdataset

In [5]:
super_df = load_dfs_subset('./DFA-data/merged_dfa_data_cleaned_2026_09_17.csv')
super_df.shape

(4250, 28)

In [6]:
# subset data to dfs questions
subset_df = super_df[DFS_QUESTION_COLS]
subset_df.shape

(4250, 20)

---

## LogisticRegression

In [7]:
# lr_param_dist = {
#     'C': loguniform(0.001, 100),  # inverse regularization, log scale is important
#     'solver': ['lbfgs', 'saga'],  # both support multinomial
# }

In [8]:
# lr_search = RandomizedSearchCV(
#     estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
#     param_distributions=lr_param_dist,
#     n_iter=40,
#     cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
#     scoring='balanced_accuracy',
#     verbose=1,
#     n_jobs=-1,
#     random_state=42
# )

In [9]:
# lr_search.fit(X_train, X__train)

In [10]:
# lr_model = lr_search.best_estimator_

In [11]:
# lr_best_score = lr_search.best_score_
# lr_best_params = lr_search.best_params_

In [12]:
# print("Best LR balanced_accuracy score:", lr_best_score) # 0.55 – 0.70 is a moderate score
# print("Best LR params:", lr_best_params)

Results from running random grid search:  
```
Best LR balanced_accuracy score: 0.5863629887114946  
Best LR params: {'C': np.float64(0.0019145111285710716), 'solver': 'saga'}
```

---

## Create train and test data

In [16]:
X = subset_df.drop('dfs20', axis=1)
y = subset_df['dfs20']

In [18]:
# note the use of `stratify=y` so every response level is represented in the test set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)
print(f'len(X)train): {len(X_train)}, len(y_train): {len(y_train)}')
print(f'len(X_test): {len(X_test)}, len(y_test): {len(y_test)}')

len(X)train): 2847, len(y_train): 2847
len(X_test): 1403, len(y_test): 1403


---

## LogisticAT

In [13]:
# lrat_param_dist = {
#     'alpha': loguniform(0.001, 100),  # regularization strength (NOT inverse like C), log scale
#     'max_iter': [1000, 5000],
# }

In [15]:
# # LogisticAT has no class_weight; pass balanced sample weights to mimic class_weight='balanced'
# lrat_sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)
# lrat_search.fit(X_train, y_train, sample_weight=lrat_sample_weight)

In [14]:
# lrat_search = RandomizedSearchCV(
#     estimator=LogisticAT(),
#     param_distributions=lrat_param_dist,
#     n_iter=40,
#     cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
#     scoring='balanced_accuracy',
#     verbose=1,
#     n_jobs=-1,
#     random_state=42
# )

In [24]:
# lrat_model = lrat_search.best_estimator_

In [25]:
# lrat_best_score = lrat_search.best_score_
# lrat_best_params = lrat_search.best_params_

In [26]:
# print("Best LRAT balanced_accuracy score:", lrat_best_score) # 0.55 – 0.70 is a moderate score
# print("Best LRAT params:", lrat_best_params)

```
Best LRAT balanced_accuracy score: 0.6116116009203096  
Best LRAT params: {'alpha': np.float64(55.51721685244721), 'max_iter': 5000}
```

---

## XGBoost

In [31]:
# xgb_param_dist = {
#     'n_estimators': randint(100, 1000),
#     'learning_rate': loguniform(0.005, 0.2),
#     'max_depth': randint(2, 7),
#     'min_child_weight': loguniform(0.5, 20),   # min weighted respondents per leaf
#     'subsample': uniform(0.6, 0.4),            # 0.6–1.0
#     'colsample_bytree': uniform(0.5, 0.5),     # 0.5–1.0
#     'gamma': loguniform(1e-3, 5),              # min loss reduction to split
#     'reg_alpha': loguniform(1e-3, 10),         # L1
#     'reg_lambda': loguniform(1e-3, 10),        # L2
# }

In [32]:
# xgb_search = RandomizedSearchCV(
#     estimator=xgb.XGBClassifier(
#         objective='multi:softprob',
#         eval_metric='mlogloss',
#         tree_method='hist',
#         random_state=42,
#         n_jobs=1,          # let the search parallelize across fits
#     ),
#     param_distributions=xgb_param_dist,
#     n_iter=40,
#     cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
#     scoring='balanced_accuracy',
#     verbose=1,
#     n_jobs=-1,
#     random_state=42
# )

In [19]:
# # XGBoost requires class labels to be the consecutive integers 0 to K−1, and the other models don't. 
# # If you'd rather keep y as 1–5 for everything else, shift the labels for XGBoost alone:
# y_train_xgb = y_train - 1
# y_test_xgb = y_test - 1

In [20]:
# # XGBoost has no class_weight; balanced sample weights mimic it
# xgb_sample_weight = compute_sample_weight(class_weight='balanced', y=y_train_xgb)
# xgb_search.fit(X_train, y_train_xgb, sample_weight=xgb_sample_weight)

In [21]:
# xgb_model = xgb_search.best_estimator_

In [22]:
# xgb_best_score = xgb_search.best_score_
# xgb_best_params = xgb_search.best_params_

In [23]:
# print("Best XGB score:", xgb_best_score)
# print("Best XGB params:", xgb_best_params)

```
Best XGB score: 0.6201436243965367  
Best XGB params: {
  'colsample_bytree': np.float64(0.8512420419935547),  
  'gamma': np.float64(0.02136716900898964),  
  'learning_rate': np.float64(0.014768002316749563),  
  'max_depth': 3,  
  'min_child_weight': np.float64(9.927051213197453),  
  'n_estimators': 507,  
  'reg_alpha': np.float64(4.497407193749844),  
  'reg_lambda': np.float64(0.11101191524846071),  
  'subsample': np.float64(0.8006065178748798)  
}
```

## Catboost

(Per Claude) To make CatBoost work with RandomizedSearchCV, there are three things you need to be aware of:  
1. Score with QWK, not the default. The default scoring is R² for the regressor and accuracy for the classifier. For the regressor, round the predictions inside the scorer so you're scoring what you'll actually report.

2. Don't pass a fixed eval_set for early stopping. Passing eval_set through fit() means every inner fold early-stops on the same validation data, which leaks and biases the search. The simpler approach is to drop early stopping during the search and tune iterations along with learning_rate.

3. Avoid thread oversubscription. Either set CatBoost's thread_count=1 with n_jobs=-1 on the search, or the other way around. Don't parallelize both.

After running a RandomizedSearchCV on both CatBoostRegressor and CatBoostClassifer, the best score's are very close. So, I will use the CatBoostClassifer to stay in line with the other models.

In [42]:
# def qwk_rounded(y_true, y_pred):
#     y_pred = np.clip(np.rint(y_pred), 1, 5).astype(int)
#     return cohen_kappa_score(y_true, y_pred, weights="quadratic")

# qwk_scorer = make_scorer(qwk_rounded)

In [43]:
# # CatBoost Regressor
# cbr_param_dist = {
#     "depth": randint(3, 8),
#     "learning_rate": loguniform(0.01, 0.2),
#     "iterations": randint(200, 1500),
#     "l2_leaf_reg": loguniform(1, 30),
#     "random_strength": uniform(0, 2),
#     "bagging_temperature": uniform(0, 1),
#     "border_count": [32, 64, 128],
# }

In [44]:
# cbr_search = RandomizedSearchCV(
#     CatBoostRegressor(loss_function="RMSE", verbose=0, random_seed=42, thread_count=1),
#     param_distributions=cbr_param_dist,
#     n_iter=60,
#     scoring=qwk_scorer,
#     cv=StratifiedKFold(5, shuffle=True, random_state=42),  # stratify on the 1–5 target
#     n_jobs=-1,
#     random_state=42,
#     refit=True,
# )
# cbr_search.fit(X, y)

In [45]:
# print("Best CBR score:", cbr_search.best_score_)
# print("Best CBR params:", cbr_search.best_params_)

```
Best CBR score: 0.8630902958323269  

Best CBR params: {  
    'bagging_temperature': np.float64(0.5581020020173412),  
    'border_count': 32,  
    'depth': 3,  
    'iterations': 641,  
    'l2_leaf_reg': np.float64(26.402883285404346),  
    'learning_rate': np.float64(0.021260639289175844),  
    'random_strength': np.float64(0.9944970117847709)  
}
```

In [46]:
# # CatBoost Classifer
# cbc_param_dist = {
#     "depth": randint(3, 8),                       # 3–7; shallow trees for small n
#     "learning_rate": loguniform(0.01, 0.2),
#     "iterations": randint(200, 1500),
#     "l2_leaf_reg": loguniform(1, 30),             # stronger regularization helps small data
#     "random_strength": uniform(0, 2),             # noise added to split scores
#     "bagging_temperature": uniform(0, 1),         # Bayesian bootstrap intensity
#     "border_count": [32, 64, 128],                # few unique values (1–5), so this barely matters
#     "auto_class_weights": [None, "Balanced", "SqrtBalanced"],
# }

In [47]:
# # use make_scorer(cohen_kappa_score, weights="quadratic"), since predictions are already class labels. 
# # predict returns a column vector, so if you hit shape errors, wrap the scorer with .ravel()
# qwk_clf = make_scorer(
#     lambda yt, yp: cohen_kappa_score(yt, yp.ravel(), weights="quadratic"))

In [48]:
# cbc_search = RandomizedSearchCV(
#     CatBoostClassifier(loss_function="MultiClass", verbose=0,random_seed=42, thread_count=1),
#     param_distributions=cbc_param_dist,
#     n_iter=60,
#     scoring=qwk_clf,
#     cv=StratifiedKFold(5, shuffle=True, random_state=42),
#     n_jobs=-1,
#     random_state=42,
#     refit=True,
# )
# cbc_search.fit(X, y)

In [49]:
# print("Best CBC score:", cbc_search.best_score_)
# print("Best CBC params:", cbc_search.best_params_)

```
Best CBC score: 0.8650632430837126  
Best CBC params: {
    'auto_class_weights': 'SqrtBalanced',  
    'bagging_temperature': np.float64(0.770967179954561),  
    'border_count': 128,  
    'depth': 5,  
    'iterations': 592,  
    'l2_leaf_reg': np.float64(11.213346937437466),  
    'learning_rate': np.float64(0.013940305267157801),  
    'random_strength': np.float64(0.8786730037315402)
}
```